# Breast Cancer Wisconsin (Рак груди): 
Классическая задача бинарной классификации. Нужно определить, является ли опухоль доброкачественной или злокачественной, на основе 30 числовых признаков, полученных из изображения клеток. Отличный пример для медицинской диагностики.

Попробуем обучить модели KNN, LogisticClassifaer, RandomForest, Adaboost, чтобы выявить, какая из модель лучше подходит для задачи. Также благодаря Adaboost выявим, какие признаки больше важны для модели.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

# Загрузка датасета
data = load_breast_cancer()

# создаем датасет с признаками
df = pd.DataFrame(data.data, columns=data.feature_names)

# добавляем таргет в датасет с признаками
df['target'] = data.target

# чтобы показать все колонки
pd.set_option('display.max_columns', None)
display(df.head())

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


## Значение 0 - злокачественная, 1 - доброкачественная.

# 1) Ознакомимся с данными, есть ли пропуски, сбалансированы ли классы? 

## Просмотрим наличие пропусков в данных.

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness error         5

In [3]:
df.isnull().sum()

mean radius                0
mean texture               0
mean perimeter             0
mean area                  0
mean smoothness            0
mean compactness           0
mean concavity             0
mean concave points        0
mean symmetry              0
mean fractal dimension     0
radius error               0
texture error              0
perimeter error            0
area error                 0
smoothness error           0
compactness error          0
concavity error            0
concave points error       0
symmetry error             0
fractal dimension error    0
worst radius               0
worst texture              0
worst perimeter            0
worst area                 0
worst smoothness           0
worst compactness          0
worst concavity            0
worst concave points       0
worst symmetry             0
worst fractal dimension    0
target                     0
dtype: int64

### В данных нет пропусков и все данные числовые, нет категориальных

# Посмотрим на то, сбалансированы классы или нет

In [4]:
sns.countplot(data = df, x='target', hue='target')

NameError: name 'sns' is not defined

### Данные сбалансированы

## 2) План

Мы обучим и сравним пять моделей:

| № | Модель                | Класс в `sklearn`            |
|---|-----------------------|------------------------------|
| 1 | KNN                   | `KNeighborsClassifier`       |
| 2 | Логистическая регрессия | `LogisticRegression`       |
| 3 | Решающее дерево       | `DecisionTreeClassifier`     |
| 4 | Случайный лес         | `RandomForestClassifier`     |
| 5 | AdaBoost              | `AdaBoostClassifier`         |

# KNN
Для KNN обязательно нужно нормализовать данные, потому что эта модель работает за счет расстояния признаков.

С помощью поиска по сетке можно будет найти лучшие гиперпараметры

In [ ]:
from sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [ ]:
RANDOM_STATE = 101

In [ ]:
X = df.drop('target', axis=1)
y = df['target']
X.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE)
#scaler = StandardScaler()

#X_scaler_train = scaler.fit_transform(X_train)
#X_scaler_test  = scaler.transform(X_test)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier()),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=101)

param_grid = {
    'knn__n_neighbors': list(range(1, 25)), 
    'knn__weights' : ['uniform', 'distance'],
    "knn__p": [1, 2], 
}     # манхэттенское и евклидово расстояние

full_model = GridSearchCV(pipe, param_grid, scoring='accuracy')

In [ ]:
full_model.fit(X_train, y_train)

In [ ]:
full_model.best_estimator_.get_params()

In [ ]:
full_model.cv_results_['mean_test_score']

In [ ]:
full_model.best_params_['knn__n_neighbors']

### Узнаем результат лучшей модели

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, classification_report, RocCurveDisplay, PredictionErrorDisplay 

In [ ]:
y_pred_1 = full_model.predict(X_test)

In [ ]:
confusion_matrix(y_test, y_pred_1)

In [ ]:
ConfusionMatrixDisplay.from_estimator(full_model, X_test,y_test)

In [ ]:
print(classification_report(y_test,y_pred_1))

# LogisticRegression

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('log', LogisticRegression()),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=101)


param_grid = {
    'log__solver': ['saga'],
    'log__penalty': ['l1', 'l2', 'elasticnet'],
    'log__l1_ratio': [0.1, 0.5, 0.9],   # используется только при elasticnet
    'log__C': [0.01, 0.1, 1, 10],
    'log__max_iter': [5000],
    'log__class_weight': ['balanced'],
    'log__random_state': [101],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=101)

full_model_logistic = GridSearchCV(estimator=pipe, cv=cv, param_grid=param_grid, scoring='accuracy')

In [ ]:
full_model_logistic.fit(X_train, y_train)

In [ ]:
full_model_logistic.best_estimator_.get_params()

In [ ]:
y_pred_log = full_model_logistic.predict(X_test)
confusion_matrix(y_test, y_pred_log)

In [ ]:
ConfusionMatrixDisplay.from_estimator(full_model_logistic, X_test,y_test)

In [ ]:
print(classification_report(y_test,y_pred_log))

# DecisionTreeClassifier and RandomForest

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
pipe = Pipeline([
    ('decision', DecisionTreeClassifier()),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=101)

param_grid = {
    'decision__random_state': [101],
    'decision__class_weight': ['balanced']
}
full_model_decision = GridSearchCV(estimator=pipe, param_grid=param_grid, cv=cv, scoring='accuracy')

In [ ]:
full_model_decision.fit(X_train, y_train)

In [ ]:
y_pred_decision = full_model_decision.predict(X_test)
confusion_matrix(y_test, y_pred_decision)

In [ ]:
ConfusionMatrixDisplay.from_estimator(full_model_decision, X_test,y_test)

In [ ]:
print(classification_report(y_test,y_pred_decision))

# AdaBoostClassifier

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
import numpy as np

In [ ]:
ad = AdaBoostClassifier()

param_grid = {
    'n_estimators':  [10, 25, 50, 75, 100, 150, 200],
    'learning_rate': [0.01, 0.1, 0.5, 1.0],
    'estimator':     [DecisionTreeClassifier(max_depth=1)],
    'random_state':  [101],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=101)

finall_model_ad = GridSearchCV(
    ad, param_grid, cv=cv, scoring='accuracy', n_jobs=-1
)

finall_model_ad.fit(X_train, y_train)
print(finall_model_ad.best_params_, finall_model_ad.best_score_)

In [ ]:
finall_model_ad.fit(X_train, y_train)


In [ ]:
y_pred_ad = finall_model_ad.predict(X_test)
confusion_matrix(y_test, y_pred_ad)

In [ ]:
ConfusionMatrixDisplay.from_estimator(finall_model_ad, X_test,y_test)

In [ ]:
print(classification_report(y_test,y_pred_ad))

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_pred_ad)

In [ ]:
from sklearn.metrics import accuracy_score, PrecisionRecallDisplay

In [ ]:
error_rates = []

for n in range(1, 96):
    model = AdaBoostClassifier(n_estimators=n)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    error = 1 - accuracy_score(y_test, preds)
    error_rates.append(error)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8), dpi=200)

ax.plot(range(1, 96), error_rates)
ax.set_xlim(1, 96)
ax.set_xticks(range(1, 96, 2))
ax.grid()

# посмотрим, какие признаки adaboost считает лучшими

In [ ]:
feats = pd.DataFrame(index=X.columns, data=model.feature_importances_, columns=['Важность'])
imp_feat = feats[feats['Важность'] > 0]
imp_feat.sort_values('Важность', ascending=False, inplace=True)

In [ ]:

fig, ax = plt.subplots(figsize=(12, 8), dpi=200)
sns.barplot(data=imp_feat, x=imp_feat.index, y='Важность', ax=ax)
plt.xticks(ticks=imp_feat.index,rotation=90);

### Именно такие признаки имеют большую роль по мнению adaboost